In [ ]:
import os
import shutil
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

load_dotenv(project_root / ".env", override=True)

cli_directory = Path(
    r"C:\Users\maxkr\.vscode\extensions\databricks.databricks-2.14.1-win32-x64\bin"
)

assert cli_directory.exists(), f"CLI-Ordner nicht gefunden: {cli_directory}"

os.environ["PATH"] = (
    str(cli_directory)
    + os.pathsep
    + os.environ.get("PATH", "")
)

print("CLI:", shutil.which("databricks"))
print("Profil:", os.getenv("DATABRICKS_CONFIG_PROFILE"))
print("Serverless:", os.getenv("DATABRICKS_SERVERLESS_COMPUTE_ID"))

In [ ]:
from databricks.connect import DatabricksSession

spark = (
    DatabricksSession.builder
    .serverless()
    .profile("DEFAULT")
    .getOrCreate()
)

print("Spark-Verbindung hergestellt")

### Check der Datensätze im Schema

In [ ]:
for catalog in ["workspace", "main"]:
    print(f"\nSchemas in {catalog}:")
    spark.sql(f"SHOW SCHEMAS IN {catalog}").show(100, truncate=False)


In [ ]:
tables_df = spark.sql("""
    SHOW TABLES IN main.dbdemos_retail_c360
""")

tables_df.show(100, truncate=False)

Die Datensätze sind als csv files abgelegt

In [ ]:
users_path = "/Volumes/main/dbdemos_retail_c360/c360/users"
orders_path = "/Volumes/main/dbdemos_retail_c360/c360/orders"
events_path = "/Volumes/main/dbdemos_retail_c360/c360/events"

users_df = spark.read.json(users_path)

orders_df = spark.read.json(orders_path)

events_df = spark.read.csv(
    events_path,
    header=True,
    inferSchema=True
)

## Users Dataframe

In [ ]:
users_df.show()

In [ ]:
from pyspark.sql import functions as F

print(f"Churn distribution:\n{users_df.groupBy('churn').count().show()}")
print(f"Distinct IDs:\n{users_df.select(
    F.countDistinct("id").alias("distinct_ids")
).first()["distinct_ids"]}")
print(f"Distinct emails:\n{users_df.select(
    F.countDistinct("email").alias("distinct_emails")
).first()["distinct_emails"]}")
print(f"Countries:\n{users_df.select(
    F.countDistinct("country").alias("distinct_countries")
).first()["distinct_countries"]}")
print(f"Countries:\n{users_df.select("country").distinct().show(truncate=False)}")

## Orders Dataframe

In [ ]:
orders_df.show()

In [ ]:
print(f"Distinct User IDs:\n{orders_df.select(
    F.countDistinct("user_id").alias("user_ids_distinct")
).first()["user_ids_distinct"]}")

print(f"Distinct IDs:\n{orders_df.select(
    F.countDistinct("id").alias("ids_distinct")
).first()["ids_distinct"]}")


## Events Dataframe

In [ ]:
events_df.show()

In [ ]:
print(f"Count per Platform:\n{events_df.groupBy('platform').count().show(truncate=False)}")
print(f"Count per Action:\n{events_df.groupBy('action').count().show(truncate=False)}")

## Check for Null Values

In [ ]:
import sys

sys.path.append("../src")

from utils import get_missing_values

In [ ]:
print(f"Users Dataframe:\n{get_missing_values(users_df).show(truncate=False)}")
print(f"Order Dataframe:\n{get_missing_values(orders_df).show(truncate=False)}")
print(f"Events Dataframe:\n{get_missing_values(events_df).show(truncate=False)}")
